In [2]:
!pip install -q torch torchvision tqdm matplotlib torchmetrics kagglehub albumentations
# Standard imports
import os
import gc
import math
import random
import warnings
import time
import datetime
from pathlib import Path
from tqdm.auto import tqdm
from copy import deepcopy
from collections import defaultdict, deque

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
from torch.cuda.amp import autocast, GradScaler
from torchvision.ops import generalized_box_iou, box_iou

# Metrics and visualization
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from scipy.optimize import linear_sum_assignment

# Data augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Set device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

Using device: cuda


In [4]:
# Kaggle authentication and dataset download
import kagglehub
kagglehub.login()
dataset_path = kagglehub.dataset_download('lubainamalvi1/surferspotting2')
dataset_path = Path(dataset_path)

# Explore and confirm folder names
print("Available folders:", list(dataset_path.glob("*")))

# Set proper paths based on actual folder names
base = dataset_path / "Surfer-Spotting-2"
train_img_dir = base / "train/images"
train_ann_dir = base / "train/labels"
val_img_dir = base / "valid/images"
val_ann_dir = base / "valid/labels"

print(f"Train images: {len(list(train_img_dir.glob('*.jpg')))} files")
print(f"Train labels: {len(list(train_ann_dir.glob('*.txt')))} files")
print(f"Val images: {len(list(val_img_dir.glob('*.jpg')))} files")
print(f"Val labels: {len(list(val_ann_dir.glob('*.txt')))} files")

Available folders: [PosixPath('/root/.cache/kagglehub/datasets/lubainamalvi1/surferspotting2/versions/1/Surfer-Spotting-2')]
Train images: 31104 files
Train labels: 31104 files
Val images: 2960 files
Val labels: 2960 files


In [22]:
# Custom Transformer-based Surfer Detector

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.models import resnet50, ResNet50_Weights


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerSurferDetector(nn.Module):
    def __init__(self, num_classes=1, num_queries=100, hidden_dim=256, use_resnet_backbone=False):
        super().__init__()
        self.num_classes = num_classes
        self.num_queries = num_queries
        self.hidden_dim = hidden_dim

        # Backbone
        if use_resnet_backbone:
          backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
          self.backbone = nn.Sequential(*list(backbone.children())[:-2])
          self.backbone_channels = 2048
        else:
            # Custom lightweight backbone
            self.backbone = nn.Sequential(
                nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1),

                # Stage 1
                nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(128),
                nn.ReLU(inplace=True),
                nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(128),
                nn.ReLU(inplace=True),

                # Stage 2
                nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(256),
                nn.ReLU(inplace=True),
                nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(256),
                nn.ReLU(inplace=True),

                # Stage 3
                nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(512),
                nn.ReLU(inplace=True),
                nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(512),
                nn.ReLU(inplace=True),
            )
            self.backbone_channels = 512

        # Projection from backbone to transformer dimensions
        self.input_proj = nn.Conv2d(self.backbone_channels, hidden_dim, kernel_size=1)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=8,
            dim_feedforward=2048,
            dropout=0.1,
            activation='relu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=3)

        # Transformer decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim,
            nhead=8,
            dim_feedforward=2048,
            dropout=0.1,
            activation='relu',
            batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=3)

        # Learnable queries
        self.query_embed = nn.Embedding(num_queries, hidden_dim)

        # Positional encoding
        self.pos_encoder = PositionalEncoding(hidden_dim)

        # Detection heads
        self.class_embed = nn.Linear(hidden_dim, num_classes + 1)  # +1 for background/no-object
        self.bbox_embed = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 4),  # (cx, cy, w, h)
            nn.Sigmoid()  # Ensure outputs are normalized [0-1]
        )

    def forward(self, x, targets=None):
        # Extract features from the backbone
        features = self.backbone(x)

        # Project features to transformer dimensions
        h = self.input_proj(features)

        # Reshape features for transformer: [batch_size, seq_len, hidden_dim]
        batch_size, c, h_size, w_size = h.shape
        h = h.flatten(2).permute(0, 2, 1)  # [batch_size, h*w, hidden_dim]

        # Add positional encodings
        pos_encoding = self.pos_encoder(h)

        # Transformer encoder
        memory = self.transformer_encoder(pos_encoding)

        # Prepare query embeddings
        queries = self.query_embed.weight.unsqueeze(0).repeat(batch_size, 1, 1)

        # Transformer decoder
        # Decoder expects: [batch_size, num_queries, hidden_dim]
        # Memory is: [batch_size, h*w, hidden_dim]
        outputs = self.transformer_decoder(queries, memory)

        # Predict classes and boxes
        outputs_class = self.class_embed(outputs)  # [batch_size, num_queries, num_classes+1]
        outputs_coord = self.bbox_embed(outputs)   # [batch_size, num_queries, 4] in (cx, cy, w, h) format

        # Format outputs
        predictions = {
            'pred_logits': outputs_class,
            'pred_boxes': outputs_coord  # Clearly in (cx, cy, w, h) format
        }

        # During training, compute the loss
        if self.training and targets is not None:
            loss_dict = self.compute_loss(predictions, targets, x.shape[-2:])
            return {'loss': loss_dict['loss'], 'loss_components': loss_dict}
        else:
            # During inference, decode outputs
            return self.process_detections(predictions, x.shape[-2:])

    def compute_loss(self, outputs, targets, img_size):
        """
        Compute losses between predictions and targets.

        Args:
            outputs: Dict containing 'pred_logits' and 'pred_boxes' in (cx, cy, w, h) format
            targets: List of dicts with 'boxes' in (x1, y1, x2, y2) format and 'labels'
            img_size: Original image dimensions (height, width)

        Returns:
            Dict of losses
        """
        device = outputs['pred_logits'].device
        batch_size = len(targets)
        num_queries = outputs['pred_logits'].shape[1]

        # Initialize total loss and components
        loss = torch.tensor(0.0, device=device, requires_grad=True)
        loss_class = torch.tensor(0.0, device=device, requires_grad=True)
        loss_boxes = torch.tensor(0.0, device=device, requires_grad=True)
        loss_giou = torch.tensor(0.0, device=device, requires_grad=True)

        # Track samples with objects
        samples_with_objects = 0

        for b in range(batch_size):
            # Get predictions for this sample
            pred_logits = outputs['pred_logits'][b]  # [num_queries, num_classes+1]
            pred_boxes = outputs['pred_boxes'][b]    # [num_queries, 4] in (cx, cy, w, h) format

            # Convert pred_boxes to xyxy format for loss calculation
            pred_boxes_xyxy = box_cxcywh_to_xyxy(pred_boxes)

            # Get ground truth boxes (already in xyxy format)
            if 'boxes' in targets[b] and len(targets[b]['boxes']) > 0:
                gt_boxes = targets[b]['boxes']       # [num_gt, 4] in (x1, y1, x2, y2) format
                gt_labels = targets[b]['labels']     # [num_gt]
                num_gt = len(gt_boxes)
                has_objects = num_gt > 0
            else:
                gt_boxes = torch.zeros((0, 4), device=device)
                gt_labels = torch.zeros(0, dtype=torch.long, device=device)
                num_gt = 0
                has_objects = False

            # Classification targets: 0 for background, 1 for objects
            tgt_classes = torch.zeros(num_queries, dtype=torch.long, device=device)

            if has_objects:
                samples_with_objects += 1

                # Calculate cost matrix for matcher
                cost_class = -pred_logits[:, 1:]  # Consider non-background scores

                # Calculate box costs - use xyxy format for both
                cost_bbox = torch.cdist(pred_boxes_xyxy, gt_boxes, p=1)

                # GIoU cost - both boxes already in xyxy format
                cost_giou = -generalized_box_iou(pred_boxes_xyxy, gt_boxes)

                # Combine costs (weighted)
                C = cost_bbox + 5.0 * cost_giou + 2.0 * cost_class.mean(dim=1).unsqueeze(1)

                # Hungarian matching
                C_np = C.cpu().detach().numpy()
                from scipy.optimize import linear_sum_assignment
                src_idx, tgt_idx = linear_sum_assignment(C_np)

                # Convert to tensors
                src_idx = torch.as_tensor(src_idx, dtype=torch.int64, device=device)
                tgt_idx = torch.as_tensor(tgt_idx, dtype=torch.int64, device=device)

                # Set matched query targets to 1 (surfer)
                tgt_classes[src_idx] = 1

                # Calculate box loss for matched pairs
                matched_pred_boxes = pred_boxes[src_idx]        # (cx, cy, w, h)
                matched_pred_boxes_xyxy = pred_boxes_xyxy[src_idx]  # (x1, y1, x2, y2)
                matched_gt_boxes = gt_boxes[tgt_idx]         # (x1, y1, x2, y2)

                # Convert gt_boxes to cxcywh for L1 loss calculation
                matched_gt_boxes_cxcywh = box_xyxy_to_cxcywh(matched_gt_boxes)

                # Box L1 loss - compare in same format (cxcywh)
                loss_boxes = loss_boxes + F.l1_loss(matched_pred_boxes, matched_gt_boxes_cxcywh, reduction='mean')

                # if has_objects and b == 0:  # Print only for first batch item
                #   print(f"\nDEBUG BOX INFO:")
                #   print(f"- GT boxes shape: {gt_boxes.shape}, range: [{gt_boxes.min().item()}, {gt_boxes.max().item()}]")
                #   print(f"- Pred boxes shape: {pred_boxes.shape}, range: [{pred_boxes.min().item()}, {pred_boxes.max().item()}]")
                #   print(f"- Pred boxes XYXY range: [{pred_boxes_xyxy.min().item()}, {pred_boxes_xyxy.max().item()}]")

                #   # After matching
                #   print(f"- Match count: {len(src_idx)}")
                #   if len(src_idx) > 0:
                #       print(f"- Sample matched GT box (XYXY): {matched_gt_boxes[0].tolist()}")
                #       print(f"- Sample matched GT box (CXCYWH): {matched_gt_boxes_cxcywh[0].tolist()}")
                #       print(f"- Sample matched pred box (CXCYWH): {matched_pred_boxes[0].tolist()}")

                #       single_pair_loss = F.l1_loss(
                #       matched_pred_boxes[0:1],
                #       matched_gt_boxes_cxcywh[0:1],
                #       reduction='sum'
                #       )
                #       print(f"- Single pair L1 loss: {single_pair_loss.item()}")

                #       # Calculate total L1 loss manually
                #       total_l1 = torch.abs(matched_pred_boxes - matched_gt_boxes_cxcywh).sum()
                #       print(f"- Total L1 (sum): {total_l1.item()}")
                #       print(f"- Total L1 (mean): {total_l1.item() / (len(matched_pred_boxes) * 4)}")

                # GIoU loss - use xyxy format
                loss_giou = loss_giou + (1 - torch.diag(generalized_box_iou(matched_pred_boxes_xyxy, matched_gt_boxes))).mean()

            # Classification loss
            # Add higher weight to positive class to handle imbalance
            weight = torch.tensor([0.3, 1.0], device=device)
            loss_class = loss_class + F.cross_entropy(pred_logits, tgt_classes, weight=weight)

        # Normalize losses
        loss_class = loss_class / batch_size

        loss_boxes = loss_boxes / max(samples_with_objects, 1)
        loss_giou = loss_giou / max(samples_with_objects, 1)

        # Combine with appropriate weights
        loss = 2.0 * loss_class + 10.0 * loss_boxes + 15.0 * loss_giou

        # Return loss components for monitoring
        return {
            'loss': loss,
            'loss_class': loss_class,
            'loss_boxes': loss_boxes,
            'loss_giou': loss_giou
        }
    def process_detections(self, outputs, img_size):
        """
        Process model outputs into final detections.

        Args:
            outputs: Dict with 'pred_logits' and 'pred_boxes' in (cx, cy, w, h) format
            img_size: Image dimensions (height, width)

        Returns:
            Dict with processed boxes, scores and labels
        """
        pred_logits = outputs['pred_logits'][0]  # [num_queries, num_classes+1]
        pred_boxes = outputs['pred_boxes'][0]    # [num_queries, 4] in (cx, cy, w, h) format

        # Get classification probabilities
        prob = F.softmax(pred_logits, dim=-1)

        # Get scores for the surfer class (class 1)
        scores = prob[:, 1]

        # Apply score threshold
        score_threshold = 0.3 if not self.training else 0.05
        keep = torch.where(scores > score_threshold)[0]

        # If no detections, return empty result
        if len(keep) == 0:
            return {
                'boxes': torch.zeros((0, 4), device=pred_boxes.device),
                'scores': torch.zeros(0, device=pred_boxes.device),
                'labels': torch.zeros(0, dtype=torch.int64, device=pred_boxes.device)
            }

        # Get filtered predictions
        filtered_boxes = pred_boxes[keep]    # (cx, cy, w, h) format
        filtered_scores = scores[keep]

        # Convert to xyxy format for visualization and NMS
        boxes_xyxy = box_cxcywh_to_xyxy(filtered_boxes)

        # Scale to image dimensions
        height, width = img_size
        scale_fct = torch.tensor([width, height, width, height], device=boxes_xyxy.device)
        boxes_xyxy = boxes_xyxy * scale_fct

        # Ensure boxes are within image bounds
        boxes_xyxy[:, 0].clamp_(0, width)   # x1
        boxes_xyxy[:, 1].clamp_(0, height)  # y1
        boxes_xyxy[:, 2].clamp_(0, width)   # x2
        boxes_xyxy[:, 3].clamp_(0, height)  # y2

        # Apply NMS
        keep_idx = torchvision.ops.nms(boxes_xyxy, filtered_scores, iou_threshold=0.5)

        # Return final detections
        return {
            'boxes': boxes_xyxy[keep_idx],
            'scores': filtered_scores[keep_idx],
            'labels': torch.ones_like(filtered_scores[keep_idx], dtype=torch.int64)
        }

def box_cxcywh_to_xyxy(x):
    """Convert bounding boxes from (cx, cy, w, h) to (x1, y1, x2, y2) format."""
    x_c, y_c, w, h = x.unbind(-1)
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return torch.stack(b, dim=-1)

def box_xyxy_to_cxcywh(x):
    """Convert bounding boxes from (x1, y1, x2, y2) to (cx, cy, w, h) format."""
    x0, y0, x1, y1 = x.unbind(-1)
    b = [(x0 + x1) / 2, (y0 + y1) / 2,
         (x1 - x0), (y1 - y0)]
    return torch.stack(b, dim=-1)

def generalized_box_iou(boxes1, boxes2):
    """
    Compute the generalized IoU between two sets of boxes.

    Args:
        boxes1: (N, 4) in (x1, y1, x2, y2) format
        boxes2: (M, 4) in (x1, y1, x2, y2) format

    Returns:
        giou: (N, M) tensor of pairwise GIoU values
    """
    # Use torchvision's implementation
    return torchvision.ops.generalized_box_iou(boxes1, boxes2)

# Cell 5: Dataset Class with Fixed Transform Handling
class SurferDetectionDataset(Dataset):
    def __init__(self, img_dir, ann_dir, transform=None, image_size=(640, 640)):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transform = transform
        self.image_size = image_size

        # Get all image files
        self.img_files = sorted(list(Path(img_dir).glob("*.jpg")))

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        # Load image
        img_path = self.img_files[idx]
        try:
            image = Image.open(img_path).convert("RGB")
            orig_width, orig_height = image.size
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a dummy sample on failure
            return torch.zeros((3, *self.image_size)), {"boxes": torch.zeros((0, 4)), "labels": torch.zeros(0, dtype=torch.int64)}

        # Load annotation (YOLO format: class x_center y_center width height)
        ann_path = Path(str(img_path).replace('/images/', '/labels/')).with_suffix('.txt')
        boxes = []
        labels = []

        if ann_path.exists():
            try:
                with open(ann_path, 'r') as f:
                    for line in f.readlines():
                        try:
                            data = line.strip().split()
                            if len(data) < 5:  # Skip incomplete lines
                                continue

                            class_id = int(data[0])  # Should be 0 for surfer

                            # YOLO format to normalized coordinates
                            x_center, y_center = float(data[1]), float(data[2])
                            box_width, box_height = float(data[3]), float(data[4])

                            # Skip invalid boxes
                            if box_width <= 0 or box_height <= 0:
                                continue

                            # Convert to [x_min, y_min, x_max, y_max] format while keeping normalized [0-1]
                            x_min = max(0, x_center - box_width / 2)
                            y_min = max(0, y_center - box_height / 2)
                            x_max = min(1, x_center + box_width / 2)
                            y_max = min(1, y_center + box_height / 2)

                            # Skip boxes with invalid coordinates
                            if not (x_max > x_min and y_max > y_min):
                                continue

                            boxes.append([x_min, y_min, x_max, y_max])
                            labels.append(class_id)
                        except (ValueError, IndexError) as e:
                            # Skip problematic lines
                            continue
            except Exception as e:
                print(f"Error reading annotation {ann_path}: {e}")
                # Continue with empty boxes

        # Apply transformations to image
        try:
            from_numpy_image = np.array(image)
            transformed = self.transform(image=from_numpy_image)
            image = transformed["image"]  # This is a torch tensor [C,H,W]
        except Exception as e:
            print(f"Transform error on image {img_path.name}: {e}")
            # Fall back to simple conversion
            from torchvision import transforms as T
            image = T.Resize(self.image_size)(image)
            image = T.ToTensor()(image)

        # Convert to tensor format
        if not boxes:
            # Return empty boxes if none found
            target = {
                "boxes": torch.zeros((0, 4), dtype=torch.float32),
                "labels": torch.zeros(0, dtype=torch.int64)
            }
        else:
            target = {
                "boxes": torch.tensor(boxes, dtype=torch.float32),  # Already normalized [0-1]
                "labels": torch.tensor(labels, dtype=torch.int64)
            }

        # Store original image path for visualization
        target["image_path"] = str(img_path)

        return image, target

# Cell 6: Fixed Transforms and Utilities
def get_train_transform():
    """
    Simplified transform that avoids bbox_params issues
    """
    return A.Compose([
        A.Resize(height=224, width=224),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_valid_transform():
    """
    Simplified transform that avoids bbox_params issues
    """
    return A.Compose([
        A.Resize(height=224, width=224),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def collate_fn(batch):
    """
    Custom collate function that filters out samples with no valid images
    """
    images = []
    targets = []

    for image, target in batch:
        # Only include samples with valid images
        if image is not None and isinstance(image, torch.Tensor):
            images.append(image)
            targets.append(target)

    # If no valid samples, create a dummy batch with one empty sample
    if not images:
        # Create a dummy image and target
        dummy_image = torch.zeros((3, 224, 224))
        dummy_target = {
            "boxes": torch.zeros((0, 4), dtype=torch.float32),
            "labels": torch.zeros(0, dtype=torch.int64),
            "image_path": ""
        }
        return [dummy_image], [dummy_target]

    return images, targets

class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

# Cell 7: Enhanced Visualization Functions
def visualize_predictions(model, val_loader, device, epoch, confidence_threshold=None, max_boxes=5):
    """
    Visualize model predictions on validation images, showing both a positive and negative example.

    Args:
        model: The detection model
        val_loader: Validation data loader
        device: Device for inference
        epoch: Current epoch number
        confidence_threshold: Minimum confidence to display boxes (adaptive if None)
        max_boxes: Maximum number of boxes to display
    """
    model.eval()

    # Create directory for visualizations if it doesn't exist
    vis_dir = Path("visualizations")
    vis_dir.mkdir(exist_ok=True)

    # Use adaptive confidence threshold based on epoch
    if confidence_threshold is None:
        if epoch <= 5:
            confidence_threshold = 0.05  # Very low threshold for early epochs
        elif epoch <= 15:
            confidence_threshold = 0.1   # Low threshold for middle epochs
        else:
            confidence_threshold = 0.3   # Higher threshold for later epochs

    # Find one image with surfers and one without
    positive_example = None
    positive_target = None
    negative_example = None
    negative_target = None

    with torch.no_grad():
        for images, targets in val_loader:
            if not images:
                continue

            for i, (image, target) in enumerate(zip(images, targets)):
                if 'boxes' in target and len(target['boxes']) > 0 and positive_example is None:
                    positive_example = image
                    positive_target = target
                elif ('boxes' not in target or len(target['boxes']) == 0) and negative_example is None:
                    negative_example = image
                    negative_target = target

                if positive_example is not None and negative_example is not None:
                    break

            if positive_example is not None and negative_example is not None:
                break

    if positive_example is None:
        positive_example = images[0]
        positive_target = targets[0]

    if negative_example is None:
        # Just use the first image if we couldn't find a suitable negative example
        negative_example = images[0]
        negative_target = targets[0]

    # Create a figure with two subplots
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))

    # Process both examples
    examples = [
        (positive_example, positive_target, "With Surfers", axes[0]),
        (negative_example, negative_target, "Without Surfers", axes[1])
    ]

    for i, (image, target, title, ax) in enumerate(examples):
        try:
            # Do inference with the model
            with torch.no_grad():
                image_tensor = image.unsqueeze(0).to(device)
                predictions = model(image_tensor)

                # Move predictions to CPU
                pred_boxes = predictions['boxes'].cpu().numpy()
                pred_scores = predictions['scores'].cpu().numpy()

                # Get ground truth boxes
                gt_boxes = target['boxes'].numpy() if 'boxes' in target else np.array([])

            # Denormalize image for display
            img_np = image.permute(1, 2, 0).numpy()
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            img_np = std * img_np + mean
            img_np = np.clip(img_np, 0, 1)
            img_np = (img_np * 255).astype(np.uint8)

            # Display the image
            ax.imshow(img_np)

            # Draw ground truth boxes in green
            for box in gt_boxes:
                x1, y1, x2, y2 = box
                rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='green',
                                   facecolor='none')
                ax.add_patch(rect)

            # Add a count of ground truth boxes
            ax.text(10, 30, f"Ground Truth: {len(gt_boxes)} boxes",
                    color='white', fontsize=12,
                    bbox=dict(facecolor='green', alpha=0.7))

            # Get top confidence scores regardless of threshold
            if len(pred_scores) > 0:
                # Sort by confidence (highest first)
                top_indices = np.argsort(-pred_scores)[:10]
                top_scores = pred_scores[top_indices]

                # List top 10 confidence scores on the right side of the image
                for j, score in enumerate(top_scores[:5]):  # Show only top 5
                    ax.text(img_np.shape[1] - 150, 30 + j*20, f"Top {j+1}: {score:.4f}",
                           color='white', fontsize=10,
                           bbox=dict(facecolor='blue', alpha=0.7))

            # Filter predictions by confidence for display
            if len(pred_boxes) > 0:
                # Filter by confidence threshold
                confident_indices = pred_scores >= confidence_threshold
                filtered_boxes = pred_boxes[confident_indices]
                filtered_scores = pred_scores[confident_indices]

                # Sort by confidence (highest first)
                sorted_indices = np.argsort(-filtered_scores)
                filtered_boxes = filtered_boxes[sorted_indices]
                filtered_scores = filtered_scores[sorted_indices]

                # Limit to max_boxes
                filtered_boxes = filtered_boxes[:max_boxes]
                filtered_scores = filtered_scores[:max_boxes]

                if len(filtered_boxes) == 0 and len(pred_scores) > 0:
                    # If no boxes above threshold, show top N anyway
                    top_indices = np.argsort(-pred_scores)[:max_boxes]
                    filtered_boxes = pred_boxes[top_indices]
                    filtered_scores = pred_scores[top_indices]

                    # Add a note that these are below threshold
                    ax.text(10, 60, f"Note: Showing top {max_boxes} predictions (below threshold)",
                        color='red', fontsize=12,
                        bbox=dict(facecolor='white', alpha=0.7))

                # Draw predicted boxes in red
                for j, (box, score) in enumerate(zip(filtered_boxes, filtered_scores)):
                    x1, y1, x2, y2 = box
                    rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red',
                                       facecolor='none')
                    ax.add_patch(rect)
                    ax.text(x1, y1-5, f'{score:.2f}', color='white', fontsize=10,
                           bbox=dict(facecolor='red', alpha=0.7))

                # Add a count of all predictions vs shown predictions
                ax.text(10, 90, f"Predictions: Showing {len(filtered_boxes)} of {len(pred_scores)} boxes (conf ≥ {confidence_threshold:.3f})",
                       color='white', fontsize=12,
                       bbox=dict(facecolor='red', alpha=0.7))
            else:
                ax.text(10, 60, "No predictions",
                       color='white', fontsize=12,
                       bbox=dict(facecolor='red', alpha=0.7))

            # Set title
            image_path = target.get("image_path", "Unknown")
            image_name = os.path.basename(image_path)
            ax.set_title(f"{title}: {image_name}", fontsize=14)

            # Remove axes
            ax.axis('off')

        except Exception as e:
            ax.text(0.5, 0.5, f"Error: {str(e)}",
                   horizontalalignment='center', verticalalignment='center',
                   transform=ax.transAxes, fontsize=12, color='red')
            ax.axis('off')
            print(f"Error visualizing example {i}: {e}")

    # Add a main title
    plt.suptitle(f"Epoch {epoch} Predictions", fontsize=16)

    # Save the figure
    filename = f"visualizations/epoch_{epoch}_predictions.png"
    plt.savefig(filename, bbox_inches='tight')
    plt.close(fig)

    print(f"Visualization saved to {filename}")
    return filename

# Cell 8: Enhanced Validation Function
# Fixed validation function that includes a loss value
def validate(model, data_loader, device, epoch, confidence_threshold=0.3):
    """
    Improved validation with better metrics, including loss estimation
    """
    model.eval()

    # Initialize metrics
    total_images = 0
    total_detections = 0
    total_gt_boxes = 0
    true_positives = 0
    false_positives = 0

    # IoU metrics
    total_iou = 0.0
    matched_gt = 0

    # Pseudo loss values
    val_box_loss = 2.0
    val_cls_loss = 10.0
    val_count_loss = 15.0

    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc="Validating"):
            # Skip empty batches
            if not images:
                continue

            # Process each image individually
            for i, (image, target) in enumerate(zip(images, targets)):
                # Move data to device
                image = image.to(device)

                # Count ground truth boxes
                if 'boxes' in target and len(target['boxes']) > 0:
                    gt_boxes = target['boxes'].to(device)
                    total_gt_boxes += len(gt_boxes)
                else:
                    gt_boxes = torch.zeros((0, 4), device=device)

                # Forward pass for detection
                outputs = model(image.unsqueeze(0))

                # Process predictions
                pred_boxes = outputs['boxes']
                pred_scores = outputs['scores']

                # Count detections above threshold
                confident_detections = (pred_scores >= confidence_threshold).sum().item()
                total_detections += confident_detections

                # Calculate metrics if we have both predictions and ground truth
                if len(pred_boxes) > 0 and len(gt_boxes) > 0:
                    # Calculate IoU between predictions and ground truth
                    ious = box_iou(pred_boxes, gt_boxes)

                    # For each prediction, get maximum IoU with any ground truth box
                    max_iou, _ = ious.max(dim=1)

                    # Count true positives (IoU > 0.5)
                    tp_mask = max_iou > 0.5
                    true_positives += tp_mask.sum().item()

                    # False positives: predictions with low IoU
                    false_positives += (max_iou <= 0.5).sum().item()

                    # Average IoU for predictions above threshold
                    conf_mask = pred_scores >= confidence_threshold
                    if conf_mask.sum() > 0:
                        total_iou += max_iou[conf_mask].mean().item()
                        matched_gt += 1

                    # Estimate box loss
                    if tp_mask.sum() > 0:
                      best_matches = torch.argmax(ious, dim=1)
                      matched_gt_boxes = gt_boxes[best_matches[tp_mask]]
                      matched_pred_boxes = pred_boxes[tp_mask]

                      # Convert predicted boxes to XYXY for proper comparison
                      # Or convert ground truth to CXCYWH
                      matched_gt_boxes_cxcywh = box_xyxy_to_cxcywh(matched_gt_boxes)

                      # Now compare same formats
                      val_box_loss += F.l1_loss(matched_pred_boxes, matched_gt_boxes_cxcywh).item()
                elif len(pred_boxes) > 0:
                    # All predictions are false positives if no ground truth
                    false_positives += len(pred_boxes)

                # Estimate classification loss
                if len(pred_scores) > 0:
                    # Simplified classification loss
                    target_scores = torch.zeros_like(pred_scores)
                    if len(gt_boxes) > 0:
                        # If there are ground truth boxes, some predictions should have high scores
                        if len(ious) > 0 and ious.size(0) > 0:
                            max_ious, _ = ious.max(dim=1)
                            target_scores[max_ious > 0.5] = 1.0

                    val_cls_loss += F.binary_cross_entropy_with_logits(
                        torch.logit(pred_scores + 1e-6),  # Convert scores back to logits
                        target_scores
                    ).item()

                # Count error - penalize difference in counts
                val_count_loss += abs(len(pred_boxes) - len(gt_boxes)) * 0.1

                total_images += 1

    # Calculate precision and recall
    precision = true_positives / max(true_positives + false_positives, 1)
    recall = true_positives / max(total_gt_boxes, 1)

    # F1 score
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)

    # Average IoU
    avg_iou = total_iou / max(matched_gt, 1)

    # Normalize loss components
    val_box_loss /= max(total_images, 1)
    val_cls_loss /= max(total_images, 1)
    val_count_loss /= max(total_images, 1)

    # Total loss (weighted sum)
    val_loss = 2.0 * val_box_loss + val_cls_loss + 0.5 * val_count_loss

    # Print validation metrics
    print(f"Validation Results (Epoch {epoch}):")
    print(f"  Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
    print(f"  Mean IoU: {avg_iou:.4f}")
    print(f"  Validation Loss: {val_loss:.4f} (Box: {val_box_loss:.4f}, Cls: {val_cls_loss:.4f}, Count: {val_count_loss:.4f})")
    print(f"  Detections: {total_detections} (avg {total_detections/total_images:.2f} per image)")
    print(f"  GT boxes: {total_gt_boxes} (avg {total_gt_boxes/total_images:.2f} per image)")

    # Return metrics
    metrics = {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'miou': avg_iou,
        'avg_detections': total_detections / total_images,
        'loss': val_loss,  # Add estimated loss for tracking
        'box_loss': val_box_loss,
        'cls_loss': val_cls_loss,
        'count_loss': val_count_loss
    }

    return metrics

def box_iou(boxes1, boxes2):
    """
    Compute IoU between two sets of boxes.

    Args:
        boxes1: (N, 4) in (x1, y1, x2, y2) format
        boxes2: (M, 4) in (x1, y1, x2, y2) format

    Returns:
        iou: (N, M) tensor of pairwise IoU values
    """
    return torchvision.ops.box_iou(boxes1, boxes2)

# Cell 9: Training Function
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    loss_meter = AverageMeter()

    pbar = tqdm(data_loader, desc=f"Epoch {epoch+1}")

    for batch_idx, (images, targets) in enumerate(pbar):
        try:
            # Skip empty batches
            if not images:
                continue

            # Stack images into batch
            batch_images = torch.stack(images).to(device)
            batch_targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v
                             for k, v in t.items()} for t in targets]

            # Forward pass
            outputs = model(batch_images, batch_targets)
            loss = outputs['loss']

            # Check for NaN or infinite loss
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"Warning: NaN or Inf loss detected at batch {batch_idx}, skipping...")
                continue

            # Backward pass with gradient clipping
            optimizer.zero_grad()
            loss.backward()

            # Clip gradients to avoid explosions
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)

            optimizer.step()

            # Update statistics
            loss_meter.update(loss.item())
            pbar.set_postfix(loss=f"{loss_meter.avg:.4f}")

            # Save checkpoint periodically
            if (batch_idx + 1) % 1000 == 0:
                checkpoint = {
                    'epoch': epoch,
                    'batch': batch_idx,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': loss_meter.avg
                }
                torch.save(checkpoint, f'checkpoint_epoch_{epoch+1}_batch_{batch_idx+1}.pth')

        except Exception as e:
            print(f"Error in batch {batch_idx}: {e}")
            import traceback
            traceback.print_exc()
            # Continue with next batch
            continue

    print(f"Epoch {epoch+1} - Average loss: {loss_meter.avg:.4f}")
    return loss_meter.avg

# Cell 10: Helper function to plot metrics
def plot_metrics(metrics_history, epoch):
    """Plot training metrics"""
    plt.figure(figsize=(15, 10))

    # Plot training loss
    plt.subplot(2, 2, 1)
    plt.plot(metrics_history['train_loss'], 'b-', label='Training Loss')
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.legend()

    # Plot validation loss
    plt.subplot(2, 2, 2)
    plt.plot(metrics_history['val_loss'], 'r-', label='Validation Loss')
    plt.title('Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.legend()

    # Plot mean IoU
    plt.subplot(2, 2, 3)
    plt.plot(metrics_history['val_miou'], 'g-', label='Mean IoU')
    plt.title('Validation Mean IoU')
    plt.xlabel('Epoch')
    plt.ylabel('mIoU')
    plt.grid(True)
    plt.legend()

    # Plot avg detections
    plt.subplot(2, 2, 4)
    plt.plot(metrics_history['val_detections'], 'y-', label='Avg Detections')
    plt.title('Average Detections per Image')
    plt.xlabel('Epoch')
    plt.ylabel('Detections')
    plt.grid(True)
    plt.legend()

    plt.suptitle(f'Training Metrics (Epoch {epoch})')
    plt.tight_layout()

    # Save the plot
    metrics_dir = Path("metrics")
    metrics_dir.mkdir(exist_ok=True)
    plt.savefig(f"metrics/metrics_epoch_{epoch}.png")
    plt.close()

    print(f"Metrics plot saved to metrics/metrics_epoch_{epoch}.png")

# Create the model
def create_model(use_resnet=False):
    model = TransformerSurferDetector(
        num_classes=1,      # 1 class for surfer
        num_queries=100,    # Number of object queries
        hidden_dim=256,     # Transformer hidden dimension
        use_resnet_backbone=use_resnet  # Set to True to use ResNet50 backbone
    )
    return model

# Cell 11: Main Training Function
# Updated main function with fixed model creation and optimizer settings
def main(num_epochs=50, batch_size=16, resume_from=None):
    """
    Main training function with improved validation and visualization
    """
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    import kagglehub
    kagglehub.login()
    dataset_path = kagglehub.dataset_download('lubainamalvi1/surferspotting2')
    dataset_path = Path(dataset_path)

    # Explore and confirm folder names
    print("Available folders:", list(dataset_path.glob("*")))

    # Set proper paths based on actual folder names
    base = dataset_path / "Surfer-Spotting-2"
    train_img_dir = base / "train/images"
    train_ann_dir = base / "train/labels"
    val_img_dir = base / "valid/images"
    val_ann_dir = base / "valid/labels"

    # Create datasets with the fixed dataset class
    train_dataset = SurferDetectionDataset(
        img_dir=train_img_dir,
        ann_dir=train_ann_dir,
        transform=get_train_transform(),
        image_size=(224, 224)
    )

    val_dataset = SurferDetectionDataset(
        img_dir=val_img_dir,
        ann_dir=val_ann_dir,
        transform=get_valid_transform(),
        image_size=(224, 224)
    )

    print(f"Training dataset size: {len(train_dataset)}")
    print(f"Validation dataset size: {len(val_dataset)}")

    # Create data loaders with the safe collate function
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=1,  # Use batch_size=1 for validation to avoid OOM errors
        shuffle=True,  # Shuffle to see different images each epoch
        collate_fn=collate_fn,
        num_workers=2
    )


    # Create model - fully custom with no pre-trained components
    model = TransformerSurferDetector(
      num_classes=1,
      num_queries=100,
      hidden_dim=256,
      use_resnet_backbone=True  # Change to True
    )

    # Initialize from pre-trained weights if possible
    model.to(device)

    # Use a lower learning rate for stability
    learning_rate = 0.00005  # Reduced from 0.0001
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)

    # Add learning rate scheduler to reduce LR during training
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5,
        verbose=True
    )

    # Track metrics for plotting
    metrics_history = {
        'train_loss': [],
        'val_loss': [],
        'val_miou': [],
        'val_detections': []
    }

    # Resume training if specified
    start_epoch = 0
    best_miou = 0.0
    best_model_path = 'best_model.pth'

    if resume_from and os.path.exists(resume_from):
        try:
            checkpoint = torch.load(resume_from)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint.get('epoch', 0) + 1

            # Load metrics history if available
            if 'metrics_history' in checkpoint:
                metrics_history = checkpoint['metrics_history']

            # Get best mIoU if available
            if 'best_miou' in checkpoint:
                best_miou = checkpoint['best_miou']

            print(f"Resuming from epoch {start_epoch}, best mIoU: {best_miou:.4f}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting fresh training")

    print(f"Starting training from epoch {start_epoch + 1}...")

    # Training loop with robust error handling
    try:
        for epoch in range(start_epoch, num_epochs):
            # Train for one epoch
            train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
            metrics_history['train_loss'].append(train_loss)

            # Validate and visualize
            print(f"Validating and visualizing at epoch {epoch+1}...")
            val_metrics = validate(model, val_loader, device, epoch+1)

            # Update metrics history
            metrics_history['val_loss'].append(val_metrics['loss'])
            metrics_history['val_miou'].append(val_metrics['miou'])
            metrics_history['val_detections'].append(val_metrics['avg_detections'])

            # Update learning rate based on validation loss
            scheduler.step(val_metrics['loss'])

            # Save checkpoint
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'train_loss': train_loss,
                'val_metrics': val_metrics,
                'metrics_history': metrics_history,
                'best_miou': best_miou
            }
            torch.save(checkpoint, f'checkpoint_epoch_{epoch+1}.pth')

            # Save best model based on mIoU
            if val_metrics['miou'] > best_miou:
                best_miou = val_metrics['miou']
                torch.save(checkpoint, best_model_path)
                print(f"New best model saved with mIoU: {best_miou:.4f}")

            # Plot metrics
            plot_metrics(metrics_history, epoch+1)
            vis_filename = visualize_predictions(model, train_loader, device, epoch, confidence_threshold=0.01)
            print(f"Visualization saved to {vis_filename}")

            # Clean up memory
            gc.collect()
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"Error during training: {e}")
        import traceback
        traceback.print_exc()
        # Save emergency checkpoint
        checkpoint = {
            'epoch': epoch if 'epoch' in locals() else 0,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if 'scheduler' in locals() else None,
            'metrics_history': metrics_history
        }
        torch.save(checkpoint, 'emergency_checkpoint.pth')
        raise e

    print(f"Training complete! Best mIoU: {best_miou:.4f}")
    return model, metrics_history

# Cell 12: Test Function
def test_small_batch():
    print("Running test on a small batch...")

    # Create tiny dataset
    train_dataset = SurferDetectionDataset(
        img_dir=train_img_dir,
        ann_dir=train_ann_dir,
        transform=get_train_transform(),
        image_size=(224, 224)
    )

    # Use just 5 images
    indices = list(range(5))
    train_tiny = torch.utils.data.Subset(train_dataset, indices)

    # Test loading a single image first
    print("Testing image loading...")
    image, target = train_tiny[0]
    print(f"Successfully loaded image: shape={image.shape}")
    print(f"Number of boxes: {len(target['boxes'])}")

    # Create minimal dataloader
    train_loader = DataLoader(
        train_tiny,
        batch_size=1,  # Use batch size of 1
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0
    )

    # Create model and optimizer
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TransformerSurferDetector(num_classes=1)
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0001)  # Lower learning rate

    # Try one forward/backward pass
    print("Testing forward/backward pass...")
    model.train()

    try:
        for batch_idx, (images, targets) in enumerate(train_loader):
            if len(images) == 0:
                print("Empty batch, skipping")
                continue

            # Forward pass
            batch_images = torch.stack(images).to(device)
            batch_targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v
                             for k, v in t.items()} for t in targets]

            outputs = model(batch_images, batch_targets)
            loss = outputs['loss']

            # Check loss value
            print(f"Loss value: {loss.item()}")

            # Backward pass
            optimizer.zero_grad()
            loss.backward()

            # Clip gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)

            optimizer.step()

            print(f"Batch {batch_idx+1}: Loss = {loss.item():.4f}")

            # Test visualization
            print("Testing visualization...")
            try:
                # Use a very low threshold for the test to see detections
                vis_filename = visualize_predictions(model, train_loader, device, 0, confidence_threshold=0.01)
                print(f"Visualization saved to {vis_filename}")
            except Exception as e:
                print(f"Error during visualization: {e}")

            break  # Just test one batch

        print("Test completed successfully!")

    except Exception as e:
        print(f"Error during test: {e}")
        import traceback
        traceback.print_exc()

# Cell 13: Run Training
# Option 1: Test with a small batch first (recommended)
#test_small_batch()

# Option 2: Full training with smaller batch size
main(num_epochs=50, batch_size=16, resume_from="checkpoint_epoch_25.pth")

Using device: cuda


Available folders: [PosixPath('/root/.cache/kagglehub/datasets/lubainamalvi1/surferspotting2/versions/1/Surfer-Spotting-2')]
Training dataset size: 31104
Validation dataset size: 2960
Resuming from epoch 25, best mIoU: 0.0000
Starting training from epoch 26...


Epoch 26:   0%|          | 0/1944 [00:00<?, ?it/s]

Epoch 26 - Average loss: 14.9792
Validating and visualizing at epoch 26...


Validating:   0%|          | 0/2960 [00:00<?, ?it/s]

...

Epoch 50:   0%|          | 0/1944 [00:00<?, ?it/s]

Epoch 50 - Average loss: 14.7167
Validating and visualizing at epoch 50...


Validating:   0%|          | 0/2960 [00:00<?, ?it/s]

Validation Results (Epoch 50):
  Precision: 0.0000, Recall: 0.0000, F1: 0.0000
  Mean IoU: 0.0000
  Validation Loss: 1.2265 (Box: 0.0007, Cls: 1.1865, Count: 0.0772)
  Detections: 8186 (avg 2.77 per image)
  GT boxes: 7181 (avg 2.43 per image)
Metrics plot saved to metrics/metrics_epoch_50.png
Visualization saved to visualizations/epoch_49_predictions.png
Visualization saved to visualizations/epoch_49_predictions.png
Training complete! Best mIoU: 0.0000


(TransformerSurferDetector(
   (backbone): Sequential(
     (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
     (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (2): ReLU(inplace=True)
     (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
     (4): Sequential(
       (0): Bottleneck(
         (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
         (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
         (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
         (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
         (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
         (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
         (relu): ReLU(inplace=True)
         (downsample): Seq